In [ ]:
import pandas as pd
import os 


examplefile = "/biodata/franco/transcriptomics/qcnorm/examples/GEUVADIS_GBR20_sample_metadata.tsv"
df_ex = pd.read_table(examplefile)
df_ex

In [ ]:
required_names = ["sample_id", "genotype_id", "qtl_group", "rna_qc_passed", "genotype_qc_passed", "study", "sex", "cell_type", "condition", "timepoint", "read_length", "stranded","paired","protocol"]

In [ ]:
VERSION = "v5"

als_metadata_samples = f"/biodata/franco/datasets/answerALS/metadata_{VERSION}/all_donor_sample_ids.txt"

df_als_samples = pd.read_table(als_metadata_samples)
blacklist = ['CASE-NEUBD141MGD',
             'CASE-NEUHL096UF0',
             'CASE-NEUHL814WMU',
             'CASE-NEULM691PBK',
             'CASE-NEUVL876PUV']

In [ ]:
als_file = f"/biodata/franco/datasets/answerALS/metadata_{VERSION}/aals_dataportal_datatable_04182023.csv"
df_als = pd.read_csv(als_file)


In [ ]:
discarded_cols = ['GUID', 'NYGC_CGND_ID', 'PLS Subgroup','Number of Visits','Revised El Escorial Criteria', 'Site of Onset', 'Bulbar', 'Axial', 'Limb', 'Other',
       'Age At Symptom Onset', 'Age At Death', 'ALSFRS-R Baseline',
       'ALSFRS-R Latest', 'ALSFRS-R Progression Slope', 'CBS Baseline',
       'CBS Latest', 'CBS Progression Slope','Has iPSC', 'For Sale Cedars',
       'Has DIMN', 'root iPSC Line ID', 'iPSC Line ID', 'diMN Line Name',
       '% SMI32', '% ISL1', '% NKX61', '% TUJ1',
       '% S100b', '% Nestin', 'Staining Details',
       'ClinReport Mutations Details', 'Has Variant (WGS)',
       'C9orf72 repeat length', 'ATXN2 repeat length','Genomics', 'Epigenomics',
       'Transcriptomics', 'Proteomics']

In [ ]:
df_als.columns

In [ ]:
df_als[['Participant_ID', 'Sex', 'Ethnicity',
       'Race','Age at Sample Collection', 'Subject Group', 'Primary Tissue', 'Batch ID',
       'WGS_Sequencing_Protocol_Used']]

In [ ]:
als_file2 = f"/biodata/franco/datasets/answerALS/metadata_{VERSION}/aals_participants v2.csv"
df_als2 = pd.read_csv(als_file2)

In [ ]:
als_cohorts = df_als2[["Participant_ID", "Cohort"]]

In [ ]:
### samples with genomics and transcriptomics
df_als_samples_wgsrna = df_als_samples[df_als_samples["genomics"].notnull() & df_als_samples["transcriptomics"].notnull()]

### Samples with transcriptomics but no genomics
df_nowgs_rna = df_als_samples[df_als_samples["genomics"].isnull() & df_als_samples["transcriptomics"].notnull()]["donor_id"]

In [ ]:
## Data with genomics
pd.DataFrame.merge( df_als_samples[df_als_samples["genomics"].notnull()], als_cohorts, left_on="donor_id", right_on="Participant_ID", how='inner').groupby("Cohort").count() 

In [ ]:
## Data merge of all samples with genomics and transcriptomics
pd.DataFrame.merge( df_als_samples_wgsrna, als_cohorts, left_on="donor_id", right_on="Participant_ID", how='inner').groupby("Cohort").count() 

In [ ]:
## Data merge of sampels with transcriptomics but NO genomics
pd.DataFrame.merge( df_nowgs_rna, als_cohorts, left_on="donor_id", right_on="Participant_ID", how='inner').groupby("Cohort").count()

In [ ]:
#### Create qcnorm sample metadata file
# good columns = ['Participant_ID', 'Sex', 'Ethnicity',
#                'Race','Age at Sample Collection', 'Subject Group', 'Primary Tissue', 'Batch ID',
#                'WGS_Sequencing_Protocol_Used']

df_sample_final = pd.DataFrame.merge( df_als_samples_wgsrna, df_als[['Participant_ID', 'Sex', 'Primary Tissue']], left_on="donor_id", right_on="Participant_ID", how='inner')

print(df_sample_final['Primary Tissue'].drop_duplicates())
df_sample_final = df_sample_final.drop(columns=["epigenomics","proteomics","Participant_ID", 'Primary Tissue'])

df_sample_final['cell_type'] = "PBMC"
df_sample_final['qtl_group'] = "PBMC"
df_sample_final['condition'] = "naive"
df_sample_final['timepoint'] = 0
df_sample_final['read_length'] = "100bp"
df_sample_final['stranded'] = "TRUE"
df_sample_final['paired'] = "TRUE"
df_sample_final['protocol'] = "Ribo-Zero"
df_sample_final['rna_qc_passed'] = "TRUE"
df_sample_final['genotype_qc_passed'] = "TRUE"
df_sample_final['study'] = "AnswerALS"

df_sample_final = df_sample_final.rename(columns={'transcriptomics':'sample_id', 'genomics':'genotype_id', 'Sex':'sex'})
df_sample_final["sex"] = df_sample_final["sex"].str.lower()

In [ ]:
for i in required_names:
    if i not in df_sample_final.columns:
        print(i)
        raise
print("SUCCESS!")

df_sample_final.to_csv(f"/biodata/franco/datasets/answerALS/transcriptomics/sample_metadata_for_qcnorm_{VERSION}.txt", sep="\t", index=False)

In [ ]:
### Now go on and make a file full of covariates that can be corrected in the usual way

df_sample_final_complete = pd.DataFrame.merge( df_als_samples_wgsrna, df_als, left_on="donor_id", right_on="Participant_ID", how='inner')
df_sample_final_complete.groupby("Race").count()

In [ ]:
df_sample_final_complete.columns

In [ ]:
df_als_samples_todas = pd.DataFrame.merge( df_als_samples, df_als, left_on="donor_id", right_on="Participant_ID", how='inner')


In [ ]:
df_als_samples_todas.to_csv(f"/biodata/franco/datasets/answerALS/metadata_{VERSION}/all_samples_metadata_processed_merged.txt", sep="\t", index=False)
